# High-Level Backtesting of the Turtle Soup Strategy

In [1]:
# imports
import time
import pandas as pd
from nautilus_trader.backtest.config import BacktestVenueConfig, BacktestDataConfig, BacktestRunConfig
from nautilus_trader.backtest.engine import BacktestResult, BacktestEngine, BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.common.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model import BarType, Bar, Venue, InstrumentId
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.config import DataCatalogConfig
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.trading.config import ImportableStrategyConfig
import sys
from pathlib import Path

from core.enums import MoneyManagementType

In [2]:
sys.path.append(str(Path.cwd().parent))

catalog = ParquetDataCatalog("../catalog")

start_ns = dt_to_unix_nanos(pd.Timestamp("2025-07-09"))
end_ns = dt_to_unix_nanos(pd.Timestamp("2025-10-22"))

instrument = TestInstrumentProvider.es_future(2025, 12)
instrument_id = instrument.id.value

# Configure backtesting
venue = BacktestVenueConfig(
    name="GLBX",
    oms_type=OmsType.NETTING,
    account_type="MARGIN",
    base_currency="USD",
    starting_balances=["10_000 USD"],
)

# Configure a catalog for a live system
catalog_cfg = DataCatalogConfig(
    path=str(catalog.path),
    fs_protocol="file",
    name="local"
)

base_bar_type = BarType.from_str(f"{instrument_id}-1-MINUTE-LAST-EXTERNAL")
data = BacktestDataConfig(
    catalog_path=str(catalog.path),
    catalog_fs_protocol="file",
    data_cls=Bar,
    bar_types=[base_bar_type],
    instrument_id=instrument_id,
    start_time=start_ns,
    end_time=end_ns
)

engine = BacktestEngineConfig(
    strategies=[
        ImportableStrategyConfig(
            strategy_path="strategies.turtle_soup.turtle_soup_strategy:TurtleSoupStrategy",
            config_path="strategies.turtle_soup.turtle_soup_strategy:TurtleSoupStrategyConfig",
            config={
                "liquidity_pool_bar_type": BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "liquidity_pool_upper_period_window": 3,
                "liquidity_pool_lower_period_window": 3,

                "turtle_soup_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "turtle_soup_analysis_chain_bar_type": [BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                                                        BarType.from_str(f"{instrument_id}-15-MINUTE-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                                                        BarType.from_str(f"{instrument_id}-5-MINUTE-LAST-INTERNAL@1-MINUTE-EXTERNAL")],
                "retries_count_on_stop_out": 2,
                "turtle_soup_bars_count": 4,

                "sma_filter_bar_type": BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "sma_filter_period": 75,

                "risk_reward_ratio": 2.0,

                "money_management_type": MoneyManagementType.MIN_QUANTITY,
                "fixed_lot": 1,
                "fixed_risk_percent": 1,

                "instrument_id": instrument_id,
                "base_bar_type": base_bar_type,
                "is_backtest": True,
            },
        ),
    ],
    logging=LoggingConfig(log_level="ERROR"),
    catalogs=[catalog_cfg]
)

config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[data],
)

node = BacktestNode(configs=[config])

# run backtesting
elapsed_start = time.perf_counter()
# Runs one or many configs synchronously
results: list[BacktestResult] = node.run()
elapsed_end = time.perf_counter()

print(f"Elapsed time: {elapsed_end - elapsed_start:.6f} seconds")

Elapsed time: 7.559158 seconds


In [3]:
results

[BacktestResult(trader_id='BACKTESTER-001', machine_id='Yuriis-MacBook-Pro.local', run_config_id='24eba98667716bf73969a314a40f3d4fb5f8b808e9eeeace8c157a967dca6c8e', instance_id='c335c3fd-b20b-4de7-9210-ac4256d06064', run_id='77c98bd1-a749-4090-ab34-dc7ce74968ed', run_started=1762166488389955000, run_finished=1762166495779574000, backtest_start=1752019200000000000, backtest_end=1760918400000000000, elapsed_time=8899200.0, iterations=100467, total_events=419, total_orders=165, total_positions=1, stats_pnls={'USD': {'PnL (total)': 746.53, 'PnL% (total)': 7.465300000000006, 'Max Winner': 190.76, 'Avg Winner': 54.39375, 'Min Winner': 4.5, 'Min Loser': -1.75, 'Avg Loser': -13.751111111111111, 'Max Loser': -24.75, 'Expectancy': 29.8616, 'Win Rate': 0.64}}, stats_returns={'Returns Volatility (252 days)': 0.048777635412407434, 'Average (Return)': 0.0012983879966166126, 'Average Loss (Return)': -0.0014531474129588135, 'Average Win (Return)': 0.003132744936333564, 'Sharpe Ratio (252 days)': 6.987

In [4]:
backtest_engine: BacktestEngine = node.get_engine(config.id)
positions = backtest_engine.trader.generate_positions_report()

In [5]:
len(positions)

25

In [6]:
pd.set_option("display.max_rows", 200)   # show all rows
pd.set_option("display.max_columns", None)  # show all cols
positions

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,ts_init,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
ESZ5.GLBX-TurtleSoupStrategy-000-2c912fd6-92e9-420b-8e99-82483345e052,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250709-014000-001-000-1,O-20250709-014000-001-000-3,BUY,FLAT,0,1,1752025200000000000,2025-07-09 01:40:00+00:00,1752026280000000000,2025-07-09 01:58:00+00:00,1080000000000,6264.500000,6269.0000,[0.00 USD],0.00072,4.50 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-143474cb-6b47-45fe-963e-bb5daa767798,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250711-082500-001-000-4,O-20250722-083500-001-000-27,BUY,FLAT,0,4,1752222300000000000,2025-07-11 08:25:00+00:00,1753182420000000000,2025-07-22 11:07:00+00:00,960120000000000,6301.765625,6297.4375,[0.00 USD],-0.00069,156.25 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-18efa27f-31b6-4cf7-92d2-959e16662493,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250722-144500-001-000-28,O-20250722-150000-001-000-39,BUY,FLAT,0,4,1753195500000000000,2025-07-22 14:45:00+00:00,1753261440000000000,2025-07-23 09:04:00+00:00,65940000000000,6336.312500,6371.0625,[0.00 USD],0.00548,139.01 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-7b8d7e75-0e75-48a5-9bd0-85f2b3c76c71,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250728-093000-001-000-40,O-20250728-093000-001-000-42,BUY,FLAT,0,2,1753695060000000000,2025-07-28 09:31:00+00:00,1753915440000000000,2025-07-30 22:44:00+00:00,220380000000000,6411.593750,6426.3750,[0.00 USD],0.00231,18.25 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-8093d497-7060-4087-9e9c-383d8e2a5710,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250731-200000-001-000-52,O-20250731-200000-001-000-53,BUY,FLAT,0,1,1753992060000000000,2025-07-31 20:01:00+00:00,1753993140000000000,2025-07-31 20:19:00+00:00,1080000000000,6374.000000,6358.2500,[0.00 USD],-0.00247,-15.75 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-6b9eb956-edc2-471d-ae40-5cf17e7f775e,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250801-064500-001-000-55,O-20250801-064500-001-000-57,BUY,FLAT,0,1,1754030760000000000,2025-08-01 06:46:00+00:00,1754338140000000000,2025-08-04 20:09:00+00:00,307380000000000,6350.250000,6364.0000,[0.00 USD],0.00217,13.75 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-34b71d86-5245-42d9-975b-f1905e8f2be6,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250811-084500-001-000-58,O-20250811-084500-001-000-60,BUY,FLAT,0,1,1754901900000000000,2025-08-11 08:45:00+00:00,1754907900000000000,2025-08-11 10:25:00+00:00,6000000000000,6415.750000,6422.7500,[0.00 USD],0.00109,7.00 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-71a2ec02-2309-4f3d-b52b-bf48e4124587,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250814-124500-001-000-61,O-20250814-124500-001-000-63,BUY,FLAT,0,1,1755175560000000000,2025-08-14 12:46:00+00:00,1755223800000000000,2025-08-15 02:10:00+00:00,48240000000000,6466.750000,6501.5000,[0.00 USD],0.00537,34.75 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-096ca71f-50b4-4d86-ac1e-9c144bcb5186,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250818-065000-001-000-64,O-20250818-074000-001-000-68,BUY,FLAT,0,2,1755499800000000000,2025-08-18 06:50:00+00:00,1755504780000000000,2025-08-18 08:13:00+00:00,4980000000000,6464.750000,6463.8750,[0.00 USD],-0.00014,-1.75 USD,True
